In [ ]:
#Key findings from EDA to consider models:
#Nodes are nearly identical
#There is a strong daily pattern
#All follow the same weekly and holiday behavior as well
#Extreme spikes come during the same season (summer)

#Will start building a forecasting tool since it can lead naturally to anomaly detection because flag hours where the actual price deviates far from the forecast
#Every day, forecast the 24 hourly MDA prices of tomorrow, for each of the 9 nodes. Hourly precision is key
#model must be able to analyze and handle seasonality, and have computing, costs and training benefits.
#GOAL: What will electricity cost at each Monclova node, for each hour of tomorrow?”
#WHY?: Knowing tomorrow’s prices ahead of time lets stakeholders move flexible activities to cheaper hours, plan purchases and bids and prepare for expensive days.
#Parameters prediction is based on: Hour, weekday, holiday, historical prices
#model type: supervised learning, there are labels, historical data, training and test sets
#Important to set safe enviroments for training to avoide leakage
#model will pretty much look like: price = long term trend + daily shape + weekly shape + holiday effects + recent price
#Type of model: xgboost, a team of decision trees to backup the predictions
#


What the model may know (no leakage): when forecasting tomorrow, you know prices up to the end of today, so for any hour tomorrow, the price 24 hours earlier is known, as is everything older. All price-based features use lags of 24h or more. A model using “the price 1 hour ago” would look great in testing and be impossible in real use.

3. Split by time: train Jan 2024 – Mar 2025, test Apr – Jun 2025 (3 months, the full spring season). Caveat to state: the test period has no extreme spikes like 2024’s, so test errors will look better than in a heat-wave year.

4. Metrics:

MAE (main): average error in MXN/MWh, easy to explain.
RMSE: punishes big misses more.
Peak MAE (18:00–21:59): the expensive hours, where accuracy is worth the most.
Skill vs. baseline: % MAE improvement over the naive forecast, the executive number.
No MAPE: zero prices make it divide by zero.

5. The three models:

Naive baseline: tomorrow at hour h = today at hour h (lag_24). The bar to beat, and the production fallback.
XGBoost: one global model for all 9 nodes (node as a feature), using calendar + lag features. Justified by your EDA: the nodes share one signal.
Prophet: one model per node. A fairness detail matters here: plain Prophet would forecast all 3 test months from March 31, a 91-day horizon, while XGBoost forecasts one day ahead. That would be an unfair comparison. The fix: give Prophet the same lag features as extra inputs (add_regressor), so both models work day-ahead with the same information. We’ll do that when we get to Prophet.

In [ ]:
#importing libraries, calling the model parameters set on yaml document and the database. Yaml's objective is to set parameters there in case those change so
#the code can keep on functioning properly without needing to update it.

In [5]:
import numpy as np
import pandas as pd
import yaml
import holidays
import plotly.express as px

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

m_cfg      = cfg["modeling"]
TARGET     = m_cfg["target"]
SPLIT_DATE = pd.Timestamp(m_cfg["split_date"])
PEAK_HOURS = m_cfg["peak_hours"]
LAGS       = m_cfg["lags"]

df = pd.read_parquet("../data/processed/pml_mda.parquet")
df = df.sort_values(["clv_nodo", "timestamp"]).reset_index(drop=True)
print(df.shape, "| split at", SPLIT_DATE.date())

(118152, 8) | split at 2025-04-01


In [ ]:
#shape of the dataset looks fine, also made sure once more that my dates have such format.
#setting up the calendar features and defining separeted lag features per node. Also defining the summary of historical prices.
#Grouping price columns by node and looping through the defined lags

In [12]:
mx_holidays = holidays.Mexico(years=range(2024, 2027))

def add_features(df, lags=LAGS):
    """Calendar + lag features. All price features use data >= 24h old (day-ahead rule)."""
    df = df.sort_values(["clv_nodo", "timestamp"]).copy()
    ts = df["timestamp"]

    df["hour"]       = ts.dt.hour
    df["weekday"]    = ts.dt.day_of_week
    df["month"]      = ts.dt.month
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df["is_holiday"] = ts.dt.date.isin(mx_holidays).astype(int)

    g = df.groupby("clv_nodo")[TARGET]
    for lag in lags:
        df[f"lag_{lag}"] = g.shift(lag)

    df["roll_mean_24"] = g.transform(lambda s: s.shift(24).rolling(24).mean())
    df["roll_max_24"]  = g.transform(lambda s: s.shift(24).rolling(24).max())
    return df

feat = add_features(df)
feat[["clv_nodo", "timestamp", TARGET, "lag_24", "lag_168", "roll_mean_24"]].head(30)

,clv_nodo,timestamp,pml,lag_24,lag_168,roll_mean_24
0,06AEO-115,2024-01-01 00:00:00,279.92,NaN,NaN,NaN
1,06AEO-115,2024-01-01 01:00:00,279.21,NaN,NaN,NaN
2,06AEO-115,2024-01-01 02:00:00,274.23,NaN,NaN,NaN
3,06AEO-115,2024-01-01 03:00:00,264.01,NaN,NaN,NaN
4,06AEO-115,2024-01-01 04:00:00,243.44,NaN,NaN,NaN
5,06AEO-115,2024-01-01 05:00:00,232.46,NaN,NaN,NaN
6,06AEO-115,2024-01-01 06:00:00,0.00,NaN,NaN,NaN
7,06AEO-115,2024-01-01 07:00:00,0.00,NaN,NaN,NaN
8,06AEO-115,2024-01-01 08:00:00,0.00,NaN,NaN,NaN
9,06AEO-115,2024-01-01 09:00:00,0.00,NaN,NaN,NaN


In [ ]:
#Noticing some lag columns have no data yet which is normal since this is showing the data from first day and theer is not enough past information yet to 
#have actual data, but to be sure I'll check that only the first days have missing data to be sure.

In [7]:
feat[["lag_24", "lag_48", "lag_168", "roll_mean_24", "roll_max_24"]].isna().sum()

lag_24           216
lag_48           432
lag_168         1512
roll_mean_24     423
roll_max_24      423
dtype: int64

In [ ]:
#Data is looking as expected and making sense but will look further just to make sure

In [8]:
feat[["clv_nodo", "timestamp", TARGET, "lag_24", "lag_168", "roll_mean_24"]].iloc[168:180]

,clv_nodo,timestamp,pml,lag_24,lag_168,roll_mean_24
168,06AEO-115,2024-01-08 00:00:00,409.38,430.77,279.92,508.356250
169,06AEO-115,2024-01-08 01:00:00,381.90,406.40,279.21,503.716250
170,06AEO-115,2024-01-08 02:00:00,368.93,399.80,274.23,500.638750
171,06AEO-115,2024-01-08 03:00:00,345.82,379.98,264.01,498.171667
172,06AEO-115,2024-01-08 04:00:00,345.34,384.54,243.44,496.067917
173,06AEO-115,2024-01-08 05:00:00,359.86,380.11,232.46,493.407083
174,06AEO-115,2024-01-08 06:00:00,406.55,369.25,0.00,489.089583
175,06AEO-115,2024-01-08 07:00:00,442.24,350.97,0.00,481.558750
176,06AEO-115,2024-01-08 08:00:00,448.02,333.91,0.00,476.195417
177,06AEO-115,2024-01-08 09:00:00,389.79,298.06,0.00,471.650833


In [ ]:
#
#Spliting the data by time in train and test sets and making sure shapes look accurate. 

In [9]:
FEATURES = ["hour", "weekday", "month", "is_weekend", "is_holiday",
            "lag_24", "lag_48", "lag_168", "roll_mean_24", "roll_max_24"]

feat = feat.dropna(subset=FEATURES).reset_index(drop=True)   # drops the first week of each node

train = feat[feat["timestamp"] <  SPLIT_DATE].copy()
test  = feat[feat["timestamp"] >= SPLIT_DATE].copy()

print("Train:", train["timestamp"].min(), "→", train["timestamp"].max(), "|", len(train), "rows")
print("Test: ", test["timestamp"].min(),  "→", test["timestamp"].max(),  "|", len(test),  "rows")

Train: 2024-01-08 00:00:00 → 2025-03-31 23:00:00 | 96984 rows
Test:  2025-04-01 00:00:00 → 2025-06-30 23:00:00 | 19656 rows


In [ ]:
#Train and test data seem to have the proper shape. Gave them the common 80/20 ratio

In [11]:
def evaluate(y_true, y_pred, hours):
    err = y_true - y_pred
    peak = hours.isin(PEAK_HOURS)
    return {
        "MAE":      err.abs().mean(),
        "RMSE":     np.sqrt((err ** 2).mean()),
        "MAE_peak": err[peak].abs().mean(),
    }

def evaluate_by_node(data, pred_col):
    rows = []
    for node, g in data.groupby("clv_nodo"):
        rows.append({"clv_nodo": node, **evaluate(g[TARGET], g[pred_col], g["hour"])})
    rows.append({"clv_nodo": "ALL", **evaluate(data[TARGET], data[pred_col], data["hour"])})
    return pd.DataFrame(rows).set_index("clv_nodo").round(1)

results = {}  